# ADVANCED SQL ANALYSIS

**Author: Gordon Moenga**

# Introduction

The analysis focuses on extracting meaningful business insights through complex SQL queries, including aggregations, filtering, grouping, joins, subqueries, and window functions. The aim is to examine transaction performance, customer activity, transaction values, and risk related patterns.

**Import Libraries & Load the datasets**

In [2]:
import pandas as pd
import sqlite3

In [3]:
customers = pd.read_csv("customers")
transactions = pd.read_csv("transactions")

In [4]:
connect = sqlite3.connect(":memory:")

In [5]:
customers.to_sql('customers', connect, index=False)
transactions.to_sql('transactions', connect, index=False)

12000

In [6]:
customers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 12 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Customer_ID               1500 non-null   object 
 1   Customer_Name             1500 non-null   object 
 2   Age                       1500 non-null   int64  
 3   Gender                    1500 non-null   object 
 4   City                      1500 non-null   object 
 5   Customer_Segment          1500 non-null   object 
 6   Account_Type              1500 non-null   object 
 7   Tenure_Months             1500 non-null   int64  
 8   Digital_Engagement_Score  1500 non-null   float64
 9   Monthly_Income_Band       1500 non-null   object 
 10  Preferred_Channel         1500 non-null   object 
 11  Account_Status            1500 non-null   object 
dtypes: float64(1), int64(2), object(9)
memory usage: 140.8+ KB


In [7]:
transactions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12000 entries, 0 to 11999
Data columns (total 11 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Transaction_ID             12000 non-null  object 
 1   Customer_ID                12000 non-null  object 
 2   Transaction_DateTime       12000 non-null  object 
 3   Transaction_Type           12000 non-null  object 
 4   Amount_NGN                 12000 non-null  float64
 5   Channel                    12000 non-null  object 
 6   Device_Type                11904 non-null  object 
 7   Location                   11904 non-null  object 
 8   International_Transaction  12000 non-null  object 
 9   Transaction_Status         12000 non-null  object 
 10  Risk_Review_Flag           12000 non-null  object 
dtypes: float64(1), object(10)
memory usage: 1.0+ MB


## 1. Customer Transaction Behaviour by Segment

In [8]:
# How does transaction behaviour differ across customer segments?
pd.read_sql_query(
    """
    SELECT
        c.Customer_Segment,
        COUNT(t.Transaction_ID) AS Total_Transactions,
        COUNT(DISTINCT t.Customer_ID) AS Active_Customers,
        SUM(t.Amount_NGN) AS Total_Transaction_Value,
        AVG(t.Amount_NGN) AS AVG_Transaction_Value,
        SUM(t.Amount_NGN) / COUNT(DISTINCT t.Customer_ID) AS Value_Per_Active_Customer
    FROM transactions t
    JOIN customers c
        ON t.Customer_ID = c.Customer_ID
    GROUP BY c.Customer_Segment
    ORDER BY Total_Transactions DESC;
    """,
    connect
)

,Customer_Segment,Total_Transactions,Active_Customers,Total_Transaction_Value,AVG_Transaction_Value,Value_Per_Active_Customer
0,Everyday,5644,711,2.614589e+08,46325.110027,367734.066090
1,Premium,2361,295,1.077512e+08,45637.948251,365258.290915
2,Student,2289,278,1.074759e+08,46953.196300,386603.835719
3,SME,1706,216,8.379137e+07,49115.692679,387923.017176


## 2. Successful vs Unsuccessful Transactions by Channel

In [9]:
pd.read_sql_query(
    """
    SELECT
        t.Channel,
        COUNT(t.Transaction_ID) AS Total_Transactions,
        SUM(
            CASE
                WHEN t.Transaction_Status = 'Successful' THEN 1
                ELSE 0
            END) AS Successful_Transactions,
        SUM(
            CASE
                WHEN t.Transaction_Status <> 'Successful' THEN 1
                ELSE 0
            END) AS Unsuccessful_Transactions,
        ROUND(
            100.0 *SUM(
            CASE
                WHEN t.Transaction_Status = 'Successful' THEN 1
                ELSE 0
            END) / COUNT(t.Transaction_ID), 2
        ) AS Success_Rate
    FROM transactions t
    JOIN customers c
        ON t.Customer_ID = c.Customer_ID
    GROUP BY t.Channel
    ORDER BY Total_Transactions DESC;
    """,
    connect
)

,Channel,Total_Transactions,Successful_Transactions,Unsuccessful_Transactions,Success_Rate
0,Mobile App,5102,4579,523,89.75
1,POS,2393,2174,219,90.85
2,Web,1869,1698,171,90.85
3,ATM,1747,1602,145,91.70
4,USSD,889,803,86,90.33


## 3. Risk Review by Customer Segment

In [10]:
# Which customer segments have the highest risk review rates?
pd.read_sql_query(
    """
    SELECT
        c.Customer_Segment,
        COUNT(t.TRANSACTION_ID) AS Total_Transactions,
        SUM(
            CASE
                WHEN Risk_Review_Flag = 'Yes' THEN 1
                ELSE 0
            END) AS Risk_Reviews,
        ROUND(
            100.0 *  SUM(
                CASE
                    WHEN Risk_Review_Flag = 'Yes' THEN 1
                    ELSE 0
                END) / COUNT(t.Transaction_ID), 2
            ) AS Risk_Review_Rate
    FROM transactions t
    JOIN customers c
        ON t.Customer_ID = c.Customer_ID
    GROUP BY c.Customer_Segment
    ORDER BY Risk_Review_Rate DESC;
    """,
    connect
)    

,Customer_Segment,Total_Transactions,Risk_Reviews,Risk_Review_Rate
0,Premium,2361,480,20.33
1,Student,2289,452,19.75
2,SME,1706,331,19.40
3,Everyday,5644,1089,19.29


## 4. International vs Domestic Transaction Value and Risk

In [11]:
# How do international and domestic transactions differ in volume, value and risk review activity?
pd.read_sql_query(
    """
    SELECT
        International_Transaction,
        SUM(Amount_NGN) AS Total_Transaction_Value,
        COUNT(Transaction_ID) AS Total_Transactions,
        SUM(
            CASE
                WHEN Risk_Review_Flag = 'Yes' THEN 1
                ELSE 0
            END) AS Risk_Reviews,
        ROUND(
            100.0 *  SUM(
                CASE
                    WHEN Risk_Review_Flag = 'Yes' THEN 1
                    ELSE 0
                END) / COUNT(Transaction_ID), 2
            ) AS Risk_Review_Rate
    FROM transactions
    GROUP BY International_Transaction;
    """,
    connect
)

,International_Transaction,Total_Transaction_Value,Total_Transactions,Risk_Reviews,Risk_Review_Rate
0,No,5.404816e+08,11520,2175,18.88
1,Yes,1.999578e+07,480,177,36.88


## 5. Customer-Level Transaction Frequency

In [12]:
# How frequently are customers using the bank's transaction services?
pd.read_sql_query(
    """
    SELECT
        c.Customer_ID,
        c.Customer_Segment,
        SUM(t.AMOUNT_NGN) AS Total_Transaction_Value,
        COUNT(t.Transaction_ID) AS Total_Transactions,
        AVG(t.AMOUNT_NGN) AS Avg_Transaction_Value
    FROM customers c
    JOIN transactions t
        ON c.Customer_ID = t.Customer_ID
    GROUP BY
        c.Customer_ID,
        c.Customer_Segment
    ORDER BY Total_Transactions DESC;
    """,
    connect
)

,Customer_ID,Customer_Segment,Total_Transaction_Value,Total_Transactions,Avg_Transaction_Value
0,FT-C01296,Everyday,783519.64,20,39175.982000
1,FT-C00965,Student,376271.52,19,19803.764211
2,FT-C00209,SME,1063845.37,18,59102.520556
3,FT-C00351,SME,649777.49,18,36098.749444
4,FT-C00357,Everyday,1713516.28,18,95195.348889
...,...,...,...,...,...
1495,FT-C00923,Everyday,56024.67,2,28012.335000
1496,FT-C01279,Premium,287197.19,2,143598.595000
1497,FT-C00520,Everyday,208519.26,1,208519.260000
1498,FT-C00983,Everyday,24725.37,1,24725.370000


## 6. High-Value Transaction Analysis

In [13]:
# Which transaction types contribute most to high-value transactions?
# Using the Week 2 IQR upper bound of approximately NGN 117,922.86:
pd.read_sql_query(
    """
    SELECT
        Transaction_Type,
        COUNT(Transaction_ID) AS Total_Transactions,
        SUM(Amount_NGN) AS Transactions_Value,
        AVG(Amount_NGN) AS Avg_Transaction_Value
    FROM transactions
    WHERE Amount_NGN > 117922.86
    GROUP BY Transaction_Type
    ORDER BY Transactions_Value DESC;
    """,
    connect
)

,Transaction_Type,Total_Transactions,Transactions_Value,Avg_Transaction_Value
0,Transfer,718,1.761903e+08,245390.346964
1,Deposit,344,1.077435e+08,313207.860552
2,Cash Withdrawal,209,3.631795e+07,173770.098517
3,Card Purchase,203,2.988171e+07,147200.542315


## 7. Monthly Transaction Performance

In [17]:
# How does transaction activity and value change over time?
pd.read_sql_query(
    """
    SELECT
        strftime('%m', Transaction_DateTime) AS Month,
        COUNT(Amount_NGN) AS Total_Transactions,
        AVG(Amount_NGN) AS Avg_Total_Transaction,
        SUM(Amount_NGN) AS Total_Transactions_Value,
        ROUND(
            100 * SUM(
                CASE
                    WHEN Transaction_Status = 'Successful' THEN 1
                    ELSE 0
                END) / COUNT(Transaction_ID), 2
            ) AS Success_Rate
    FROM transactions
    GROUP BY Month
    ORDER BY Total_Transactions DESC;
    """,
    connect
)

,Month,Total_Transactions,Avg_Total_Transaction,Total_Transactions_Value,Success_Rate
0,03,4133,47471.817849,1.962010e+08,90.0
1,01,4133,45605.960486,1.884894e+08,90.0
2,02,3734,47077.369306,1.757869e+08,90.0


## 8. Rank Customers by Transaction Value

In [20]:
# Which customers generate the highest total transaction value?
pd.read_sql_query(
    """
    WITH CustomerTotals AS(
        SELECT
            c.Customer_ID,
            c.Customer_Name,
            c.Customer_Segment,
            COUNT(t.Transaction_ID) AS Total_Transactions,
            SUM(t.AMOUNT_NGN) AS Transaction_Value
        FROM customers c
        JOIN transactions t
            ON c.Customer_ID = t.Customer_ID
        GROUP BY 
            c.Customer_ID,
            c.Customer_Name,
            c.Customer_Segment
    )

    SELECT
        Customer_ID,
        Customer_Name,
        Customer_Segment,
        Total_Transactions,
        Transaction_Value,
        RANK () OVER (
            ORDER BY Total_Transactions DESC
        ) AS Customer_Rank
    FROM CustomerTotals
    ORDER BY Customer_Rank;
    """,
    connect
)

,Customer_ID,Customer_Name,Customer_Segment,Total_Transactions,Transaction_Value,Customer_Rank
0,FT-C01296,Fatima Adeyemi,Everyday,20,783519.64,1
1,FT-C00965,Tunde Garba,Student,19,376271.52,2
2,FT-C00209,Emeka Williams,SME,18,1063845.37,3
3,FT-C00351,Ada Garba,SME,18,649777.49,3
4,FT-C00357,Ada Garba,Everyday,18,1713516.28,3
...,...,...,...,...,...,...
1495,FT-C00923,Amina Eze,Everyday,2,56024.67,1484
1496,FT-C01279,Yusuf Mohammed,Premium,2,287197.19,1484
1497,FT-C00520,Grace Mohammed,Everyday,1,208519.26,1498
1498,FT-C00983,Daniel Ojo,Everyday,1,24725.37,1498
